# Method 3: Unsupervised Greedy MCCP Calibration

This notebook builds Mondrian Cross Conformal Prediction calibration data for the **Greedy Unsupervised** model using only the prepared `train` and `val` splits from `classification_data`.

### Workflow
1. Load `X_train / y_train` and `X_val / y_val`.
2. Perform `5`-fold random shuffled CV on `X_train / y_train`.
3. For each fold, train the two unsupervised greedy pretraining stages on the fold-fit split while monitoring reconstruction on `X_val`.
4. Train the classifier stage initialized from the pretrained encoder and select the fold-best classifier by **validation loss** on `X_val / y_val`.
5. Score the held-out fold with that fold-best classifier and save the held-out predictions.
6. Concatenate all held-out predictions into a calibration CSV and export the MCCP `qhat` CSV with `MCPTool`.


In [1]:
from pathlib import Path
import copy
import random

import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import KFold
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from mcp_tool import MCPTool

sns.set_theme(style="darkgrid")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


Using device: cpu


In [2]:
RANDOM_STATE = 0
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    

In [3]:
DATA_DIR = Path("./classification_data")
OUTPUT_DIR = Path("./greedy_unsupervised_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = OUTPUT_DIR / "mccp_fold_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HOLDOUT_DIR = OUTPUT_DIR / "mccp_heldout_folds"
HOLDOUT_DIR.mkdir(parents=True, exist_ok=True)

CALIBRATION_CSV = OUTPUT_DIR / "greedy_unsupervised_calibration_mccp.csv"
QHAT_CSV = OUTPUT_DIR / "greedy_unsupervised_qhat_mccp.csv"
FOLD_SUMMARY_CSV = OUTPUT_DIR / "greedy_unsupervised_mccp_fold_summary.csv"

BATCH_SIZE = 2048
LEARNING_RATE = 1e-3
STAGE1_EPOCHS = 500
STAGE2_EPOCHS = 500
FINAL_EPOCHS = 1000
CV_FOLDS = 5
MCP_ALPHA = 0.1

X_train = np.load(DATA_DIR / "X_train.npy").astype(np.float32)
X_val = np.load(DATA_DIR / "X_val.npy").astype(np.float32)
y_train = np.load(DATA_DIR / "y_train.npy").astype(np.int64)
y_val = np.load(DATA_DIR / "y_val.npy").astype(np.int64)

print("train:", X_train.shape, y_train.shape)
print("val:", X_val.shape, y_val.shape)
print("output_dir:", OUTPUT_DIR.resolve())


train: (1604, 7) (1604,)
val: (200, 7) (200,)
output_dir: /home/srinadb/BCU/BCUTorch/greedy_unsupervised_outputs


In [4]:
def make_prediction_dataframe(fsrq_probabilities, targets):
    fsrq_probabilities = np.asarray(fsrq_probabilities, dtype=np.float64).reshape(-1)
    fsrq_probabilities = np.clip(fsrq_probabilities, 1e-7, 1.0 - 1e-7)
    targets = np.asarray(targets, dtype=np.int64).reshape(-1)
    true_labels = np.where(targets == 1, "FSRQ", "BLL")
    return pd.DataFrame(
        {
            "true_label": true_labels,
            "BLL": 1.0 - fsrq_probabilities,
            "FSRQ": fsrq_probabilities,
        }
    )


def make_binary_metrics(y_true, probabilities):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities, dtype=np.float64).reshape(-1)
    predictions = (probabilities >= 0.5).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_true, predictions),
        "precision_weighted": precision_score(y_true, predictions, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, predictions, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, predictions, average="weighted", zero_division=0),
    }

    try:
        metrics["auc"] = roc_auc_score(y_true, probabilities)
    except ValueError:
        metrics["auc"] = np.nan

    return metrics


def make_loaders(X_fit, y_fit, X_valid, y_valid):
    ae_train_dataset = TensorDataset(torch.tensor(X_fit, dtype=torch.float32))
    ae_valid_dataset = TensorDataset(torch.tensor(X_valid, dtype=torch.float32))
    clf_train_dataset = TensorDataset(
        torch.tensor(X_fit, dtype=torch.float32),
        torch.tensor(y_fit, dtype=torch.float32).unsqueeze(1),
    )
    clf_valid_dataset = TensorDataset(
        torch.tensor(X_valid, dtype=torch.float32),
        torch.tensor(y_valid, dtype=torch.float32).unsqueeze(1),
    )

    ae_train_loader = DataLoader(ae_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    ae_valid_loader = DataLoader(ae_valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    clf_train_loader = DataLoader(clf_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    clf_valid_loader = DataLoader(clf_valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return ae_train_loader, ae_valid_loader, clf_train_loader, clf_valid_loader


def collect_probabilities(model, X):
    model.eval()
    with torch.no_grad():
        probabilities = model(torch.tensor(X, dtype=torch.float32, device=device))
    return probabilities.detach().cpu().numpy().reshape(-1)


def build_stage1_autoencoder(input_dim):
    return nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(64, input_dim),
    ).to(device)


def build_stage2_autoencoder(stage1_model, input_dim):
    model = nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, input_dim),
    ).to(device)

    with torch.no_grad():
        model[0].weight.copy_(stage1_model[0].weight)
        model[0].bias.copy_(stage1_model[0].bias)

    model[0].weight.requires_grad = False
    model[0].bias.requires_grad = False
    return model


def build_classifier_model(stage2_model, input_dim):
    model = nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1),
        nn.Sigmoid(),
    ).to(device)

    with torch.no_grad():
        model[0].weight.copy_(stage2_model[0].weight)
        model[0].bias.copy_(stage2_model[0].bias)
        model[3].weight.copy_(stage2_model[3].weight)
        model[3].bias.copy_(stage2_model[3].bias)

    model[0].weight.requires_grad = False
    model[0].bias.requires_grad = False
    model[3].weight.requires_grad = False
    model[3].bias.requires_grad = False
    return model


def train_autoencoder_stage(model, train_loader, valid_loader, epochs):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=LEARNING_RATE,
    )

    history_rows = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0

        for (features_batch,) in train_loader:
            features_batch = features_batch.to(device)

            optimizer.zero_grad(set_to_none=True)
            reconstructions = model(features_batch)
            loss = criterion(reconstructions, features_batch)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * features_batch.size(0)

        train_loss = train_loss_sum / len(train_loader.dataset)

        model.eval()
        valid_loss_sum = 0.0
        with torch.no_grad():
            for (features_batch,) in valid_loader:
                features_batch = features_batch.to(device)
                reconstructions = model(features_batch)
                loss = criterion(reconstructions, features_batch)
                valid_loss_sum += loss.item() * features_batch.size(0)

        valid_loss = valid_loss_sum / len(valid_loader.dataset)

        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
            }
        )

    history_df = pd.DataFrame(history_rows)
    last_state = copy.deepcopy(model.state_dict())
    return history_df, last_state


def train_classifier_stage(model, train_loader, valid_loader, epochs):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=LEARNING_RATE,
    )

    history_rows = []
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 1
    best_loss = np.inf
    best_metrics = None

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_probabilities = []
        train_targets = []

        for features_batch, targets_batch in train_loader:
            features_batch = features_batch.to(device)
            targets_batch = targets_batch.to(device)

            optimizer.zero_grad(set_to_none=True)
            probabilities = model(features_batch)
            loss = criterion(probabilities, targets_batch)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * targets_batch.size(0)
            train_probabilities.append(probabilities.detach().cpu().numpy())
            train_targets.append(targets_batch.detach().cpu().numpy())

        train_probabilities = np.concatenate(train_probabilities, axis=0).reshape(-1)
        train_targets = np.concatenate(train_targets, axis=0).reshape(-1).astype(int)
        train_loss = train_loss_sum / len(train_loader.dataset)
        train_metrics = make_binary_metrics(train_targets, train_probabilities)

        model.eval()
        valid_loss_sum = 0.0
        valid_probabilities = []
        valid_targets = []

        with torch.no_grad():
            for features_batch, targets_batch in valid_loader:
                features_batch = features_batch.to(device)
                targets_batch = targets_batch.to(device)

                probabilities = model(features_batch)
                loss = criterion(probabilities, targets_batch)

                valid_loss_sum += loss.item() * targets_batch.size(0)
                valid_probabilities.append(probabilities.detach().cpu().numpy())
                valid_targets.append(targets_batch.detach().cpu().numpy())

        valid_probabilities = np.concatenate(valid_probabilities, axis=0).reshape(-1)
        valid_targets = np.concatenate(valid_targets, axis=0).reshape(-1).astype(int)
        valid_loss = valid_loss_sum / len(valid_loader.dataset)
        valid_metrics = make_binary_metrics(valid_targets, valid_probabilities)

        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "train_accuracy": train_metrics["accuracy"],
                "valid_accuracy": valid_metrics["accuracy"],
                "valid_auc": valid_metrics["auc"],
                "valid_f1_weighted": valid_metrics["f1_weighted"],
            }
        )

        if valid_loss < best_loss:
            best_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            best_metrics = {
                "valid_loss": valid_loss,
                **valid_metrics,
            }

    history_df = pd.DataFrame(history_rows)
    last_state = copy.deepcopy(model.state_dict())
    return history_df, last_state, best_state, best_epoch, best_metrics

In [5]:
kfold = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

calibration_frames = []
fold_summaries = []

for fold_index, (fit_index, holdout_index) in enumerate(kfold.split(X_train), start=1):
    print(f"Fold {fold_index}/{CV_FOLDS}")
    fold_dir = MODEL_DIR / f"fold_{fold_index:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    holdout_csv = HOLDOUT_DIR / f"greedy_unsupervised_holdout_fold_{fold_index:02d}_mccp.csv"

    X_fit = X_train[fit_index]
    y_fit = y_train[fit_index]
    X_holdout = X_train[holdout_index]
    y_holdout = y_train[holdout_index]

    fit_bll = int((y_fit == 0).sum())
    fit_fsrq = int((y_fit == 1).sum())
    holdout_bll = int((y_holdout == 0).sum())
    holdout_fsrq = int((y_holdout == 1).sum())

    print(f"  Fit count: {len(fit_index)} | BLL: {fit_bll} | FSRQ: {fit_fsrq}")
    print(f"  Held-out count: {len(holdout_index)} | BLL: {holdout_bll} | FSRQ: {holdout_fsrq}")

    ae_train_loader, ae_valid_loader, clf_train_loader, clf_valid_loader = make_loaders(
        X_fit, y_fit, X_val, y_val
    )

    stage1_autoencoder = build_stage1_autoencoder(X_train.shape[1])
    stage1_history, stage1_last_state = train_autoencoder_stage(
        stage1_autoencoder,
        ae_train_loader,
        ae_valid_loader,
        STAGE1_EPOCHS,
    )
    stage1_history.to_csv(fold_dir / "stage1_history.csv", index=False)
    torch.save({"model_state_dict": stage1_last_state}, fold_dir / "stage1_last.pt")

    stage1_autoencoder.load_state_dict(stage1_last_state)
    stage2_autoencoder = build_stage2_autoencoder(stage1_autoencoder, X_train.shape[1])
    stage2_history, stage2_last_state = train_autoencoder_stage(
        stage2_autoencoder,
        ae_train_loader,
        ae_valid_loader,
        STAGE2_EPOCHS,
    )
    stage2_history.to_csv(fold_dir / "stage2_history.csv", index=False)
    torch.save({"model_state_dict": stage2_last_state}, fold_dir / "stage2_last.pt")

    stage2_autoencoder.load_state_dict(stage2_last_state)
    classifier_model = build_classifier_model(stage2_autoencoder, X_train.shape[1])
    final_history, final_last_state, final_best_state, best_epoch, best_metrics = train_classifier_stage(
        classifier_model,
        clf_train_loader,
        clf_valid_loader,
        FINAL_EPOCHS,
    )
    final_history.to_csv(fold_dir / "final_history.csv", index=False)
    torch.save({"model_state_dict": final_last_state}, fold_dir / "final_last.pt")
    torch.save(
        {
            "model_state_dict": final_best_state,
            "best_epoch": best_epoch,
            "best_metrics": best_metrics,
        },
        fold_dir / "final_best_by_val_loss.pt",
    )

    classifier_model.load_state_dict(final_best_state)
    holdout_probabilities = collect_probabilities(classifier_model, X_holdout)
    holdout_df = make_prediction_dataframe(holdout_probabilities, y_holdout)
    holdout_df.to_csv(holdout_csv, index=False)
    calibration_frames.append(holdout_df)

    holdout_metrics = make_binary_metrics(y_holdout, holdout_probabilities)
    fold_summaries.append(
        {
            "fold": fold_index,
            "fit_samples": len(fit_index),
            "heldout_samples": len(holdout_index),
            "fit_bll": fit_bll,
            "fit_fsrq": fit_fsrq,
            "heldout_bll": holdout_bll,
            "heldout_fsrq": holdout_fsrq,
            "best_epoch_by_val_loss": best_epoch,
            "best_val_loss": best_metrics["valid_loss"],
            "best_val_accuracy": best_metrics["accuracy"],
            "best_val_auc": best_metrics["auc"],
            "best_val_f1_weighted": best_metrics["f1_weighted"],
            "heldout_accuracy": holdout_metrics["accuracy"],
            "heldout_auc": holdout_metrics["auc"],
            "heldout_f1_weighted": holdout_metrics["f1_weighted"],
            "heldout_csv": str(holdout_csv),
            "best_model_path": str(fold_dir / "final_best_by_val_loss.pt"),
        }
    )

calibration_df = pd.concat(calibration_frames, ignore_index=True)
calibration_df.to_csv(CALIBRATION_CSV, index=False)

fold_summary_df = pd.DataFrame(fold_summaries)
fold_summary_df.to_csv(FOLD_SUMMARY_CSV, index=False)

calibration_targets = np.where(calibration_df["true_label"].to_numpy() == "FSRQ", 1, 0)
calibration_probabilities = calibration_df["FSRQ"].to_numpy(dtype=np.float64)
calibration_metrics = make_binary_metrics(calibration_targets, calibration_probabilities)

print("Saved calibration CSV:", CALIBRATION_CSV)
print("Saved fold summary CSV:", FOLD_SUMMARY_CSV)
print()
print("Combined calibration metrics:")
print(pd.Series(calibration_metrics))
print()

fold_summary_df

Fold 1/5
  Fit count: 1283 | BLL: 838 | FSRQ: 445
  Held-out count: 321 | BLL: 215 | FSRQ: 106
Fold 2/5
  Fit count: 1283 | BLL: 847 | FSRQ: 436
  Held-out count: 321 | BLL: 206 | FSRQ: 115
Fold 3/5
  Fit count: 1283 | BLL: 849 | FSRQ: 434
  Held-out count: 321 | BLL: 204 | FSRQ: 117
Fold 4/5
  Fit count: 1283 | BLL: 833 | FSRQ: 450
  Held-out count: 321 | BLL: 220 | FSRQ: 101
Fold 5/5
  Fit count: 1284 | BLL: 845 | FSRQ: 439
  Held-out count: 320 | BLL: 208 | FSRQ: 112
Saved calibration CSV: greedy_unsupervised_outputs/greedy_unsupervised_calibration_mccp.csv
Saved fold summary CSV: greedy_unsupervised_outputs/greedy_unsupervised_mccp_fold_summary.csv

Combined calibration metrics:
accuracy              0.901496
precision_weighted    0.904038
recall_weighted       0.901496
f1_weighted           0.902233
auc                   0.959233
dtype: float64



,fold,fit_samples,heldout_samples,fit_bll,fit_fsrq,heldout_bll,heldout_fsrq,best_epoch_by_val_loss,best_val_loss,best_val_accuracy,best_val_auc,best_val_f1_weighted,heldout_accuracy,heldout_auc,heldout_f1_weighted,heldout_csv,best_model_path
0,1,1283,321,838,445,215,106,989,0.259353,0.895,0.952738,0.895712,0.894081,0.954805,0.895003,greedy_unsupervised_outputs/mccp_heldout_folds...,greedy_unsupervised_outputs/mccp_fold_models/f...
1,2,1283,321,847,436,206,115,1000,0.254273,0.900,0.954167,0.900460,0.887850,0.963656,0.889232,greedy_unsupervised_outputs/mccp_heldout_folds...,greedy_unsupervised_outputs/mccp_fold_models/f...
2,3,1283,321,849,434,204,117,1000,0.253909,0.890,0.954048,0.889458,0.912773,0.962335,0.913342,greedy_unsupervised_outputs/mccp_heldout_folds...,greedy_unsupervised_outputs/mccp_fold_models/f...
3,4,1283,321,833,450,220,101,1000,0.258215,0.880,0.952500,0.879409,0.912773,0.954590,0.913219,greedy_unsupervised_outputs/mccp_heldout_folds...,greedy_unsupervised_outputs/mccp_fold_models/f...
4,5,1284,320,845,439,208,112,1000,0.256962,0.895,0.954167,0.894210,0.900000,0.961324,0.900201,greedy_unsupervised_outputs/mccp_heldout_folds...,greedy_unsupervised_outputs/mccp_fold_models/f...


In [6]:
qhat_builder = MCPTool.from_calibration_csv(str(CALIBRATION_CSV), alpha=MCP_ALPHA)
qhat_builder.export_qhat_csv(str(QHAT_CSV))

qhat_df = pd.read_csv(QHAT_CSV)
print("Saved qhat CSV:", QHAT_CSV)
qhat_df.head()

  Saved: greedy_unsupervised_outputs/greedy_unsupervised_qhat_mccp.csv
Saved qhat CSV: greedy_unsupervised_outputs/greedy_unsupervised_qhat_mccp.csv


,alpha,n_cal,qhat_BLL,qhat_FSRQ
0,0.0100,1604,0.856932,0.981572
1,0.0296,1604,0.724090,0.839452
2,0.0492,1604,0.648834,0.804955
3,0.0688,1604,0.584620,0.745153
4,0.0884,1604,0.510665,0.643163


In [7]:
print("Calibration CSV preview")
display(calibration_df.head())

print("Qhat rows nearest alpha =", MCP_ALPHA)
display(qhat_df.iloc[(qhat_df["alpha"] - MCP_ALPHA).abs().argsort()[:3]].copy())

Calibration CSV preview


,true_label,BLL,FSRQ
0,BLL,0.991724,0.008276
1,BLL,0.267745,0.732255
2,BLL,0.913839,0.086161
3,BLL,0.624731,0.375269
4,BLL,0.999851,0.000149


Qhat rows nearest alpha = 0.1


,alpha,n_cal,qhat_BLL,qhat_FSRQ
5,0.1000,1604,0.483877,0.542458
6,0.1080,1604,0.449254,0.526032
4,0.0884,1604,0.510665,0.643163
